# Fast Budget Optimization Test Suite

This notebook executes all test cases from `FAST_OPTIMIZATION_TEST_PLAN.md` to comprehensively validate the Excel-based fast budget optimization framework.

## Test Categories Priority

- 🔴 **P0**: Basic Functionality (Core optimization)
- 🟡 **P1**: Budget Configurations & Time Periods 
- 🟢 **P2**: Constraint Tests & Advanced Configurations
- 🟢 **P3**: Edge Cases
- 🔵 **P4**: Performance & Validation

## Setup and Data Loading

In [1]:
import logging
import pandas as pd
import numpy as np
import time
import warnings
from meridian.model import model, spec
from meridian.analysis import optimizer
from meridian.planner.adhoc_data_loader import AdhocDataLoader

# Configure logging to reduce noise
logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings('ignore')

print("✅ Imports successful")

✅ Imports successful


In [2]:
# Load test data once for all tests
excel_file_path = (
    '/Users/mariappan.subramanian/Library/CloudStorage/'
    'OneDrive-TheTradeDesk/MMM/BudgetOptimizer/mmm_input_artifacts.xlsx'
)

model_config = {
    'time_col': 'week',
    'geo_col': 'geo',
    'population_col': 'population',
    'kpi_type': 'non_revenue',
    'kpi_col': 'conversions',
    'revenue_per_kpi_col': 'revenue_per_conversion',
    'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
    'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
    'media_channels': ['Channel0', 'Channel1', 'Channel2'],
    'reach_cols': ['Channel3_reach'],
    'frequency_cols': ['Channel3_frequency'],
    'rf_spend_cols': ['Channel3_spend'],
    'rf_channels': ['Channel3']
}

print("Loading Excel data and creating inference data...")
loader = AdhocDataLoader(file_name=excel_file_path, model_config=model_config)
data = loader.build_input_data()
inference_data = loader.get_inference_data()

print(f"✅ Data loaded successfully")
print(f"   Time periods: {len(data.time)} ({data.time.values[0]} to {data.time.values[-1]})")
print(f"   Channels: {len(data.media_spend)}")
print(f"   Geos: {len(data.geo)}")

Loading Excel data and creating inference data...
✅ Data loaded successfully
   Time periods: 156 (2021-01-25 to 2024-01-15)
   Channels: 20
   Geos: 20


I0000 00:00:1756511770.259087 2615218 service.cc:148] XLA service 0x17772b710 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1756511770.259109 2615218 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1756511770.265970 2615218 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [3]:
# Helper functions for test execution and validation
def setup_model_and_optimizer():
    """Create fresh model and optimizer for each test."""
    dummy_model = model.Meridian(input_data=data, model_spec=spec.ModelSpec(), inference_data=inference_data)
    dummy_model.sample_prior(n_draws=100, seed=42)  # Consistent seed for reproducibility
    budget_optimizer = optimizer.BudgetOptimizer(dummy_model)
    return dummy_model, budget_optimizer

def extract_optimization_results(opt_results, test_name):
    """Extract key metrics from optimization results."""
    opt_data = opt_results.optimized_data

    return {
        'test_name': test_name,
        'spend': opt_data['spend'].values,
        'pct_of_spend': opt_data['pct_of_spend'].values,
        'roi_mean': opt_data['roi'].sel(metric='mean').values,
        'mroi_mean': opt_data['mroi'].sel(metric='mean').values,
        'incremental_outcome': opt_data['incremental_outcome'].sel(metric='mean').values,
        'effectiveness': opt_data['effectiveness'].sel(metric='mean').values,
        'total_budget': np.sum(opt_data['spend'].values),
        'total_incremental_outcome': np.sum(opt_data['incremental_outcome'].sel(metric='mean').values)
    }

def print_test_results(results, title):
    """Print formatted test results."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print('='*60)
    print(f"Total Budget: ${results['total_budget']:,.0f}")
    print(f"Total Incremental Outcome: {results['total_incremental_outcome']:,.0f}")
    print(f"Overall ROI: {results['total_incremental_outcome']/results['total_budget']:.3f}")
    print(f"\nSpend Allocation: {results['spend']}")
    print(f"Spend Percentages: {results['pct_of_spend']*100}")
    print(f"ROI by Channel: {results['roi_mean']}")
    print(f"mROI by Channel: {results['mroi_mean']}")

# Test results storage
test_results = []

print("✅ Helper functions defined")

✅ Helper functions defined


## Test 1.1: Default Parameters Baseline

In [ ]:
print("Running Test 1.1: Default Parameters Baseline")
print("Testing fast budget optimization with completely default parameters.")

try:
    start_time = time.time()
    dummy_model, budget_optimizer = setup_model_and_optimizer()

    # Run optimization with default parameters
    opt_results = budget_optimizer.optimize()

    execution_time = time.time() - start_time
    results = extract_optimization_results(opt_results, "Test 1.1: Default Baseline")
    results['execution_time'] = execution_time

    test_results.append(results)
    print_test_results(results, "Test 1.1: Default Parameters Baseline")

    # Validation against expected baseline
    expected_spend = [28300000, 27100000, 30000000, 25500000]
    expected_roi = [1.83, 2.51, 2.92, 5.42]

    print(f"\n📊 VALIDATION:")
    spend_diff = np.abs(results['spend'] - expected_spend)
    roi_diff = np.abs(results['roi_mean'] - expected_roi)

    print(f"Spend allocation difference: {spend_diff} (max: {np.max(spend_diff):,.0f})")
    print(f"ROI difference: {roi_diff} (max: {np.max(roi_diff):.3f})")
    print(f"Execution time: {execution_time:.2f} seconds")

    # Check if all metric statistics are identical (point estimates)
    opt_data = opt_results.optimized_data
    for channel in range(len(results['roi_mean'])):
        roi_stats = opt_data['roi'].sel(channel=f'Channel{channel}')
        mean_val = roi_stats.sel(metric='mean').values
        median_val = roi_stats.sel(metric='median').values
        ci_lo = roi_stats.sel(metric='ci_lo').values
        ci_hi = roi_stats.sel(metric='ci_hi').values

        if np.allclose([mean_val, median_val, ci_lo, ci_hi], mean_val):
            print(f"✅ Channel{channel}: All metrics identical (point estimates)")
        else:
            print(f"❌ Channel{channel}: Metrics differ - not using point estimates")

    print("✅ Test 1.1 completed successfully")

except Exception as e:
    print(f"❌ Test 1.1 failed: {str(e)}")
    test_results.append({'test_name': 'Test 1.1: Default Baseline', 'status': 'FAILED', 'error': str(e)})

Running Test 1.1: Default Parameters Baseline
Testing fast budget optimization with completely default parameters.

Test 1.1: Default Parameters Baseline
Total Budget: $110,900,000
Total Incremental Outcome: 345,741,440
Overall ROI: 3.118

Spend Allocation: [28300000 27100000 30000000 25500000]
Spend Percentages: [25.51848512 24.43642922 27.05139766 22.99368801]
ROI by Channel: [1.8300712 2.5073638 2.9244485 5.4222517]
mROI by Channel: [1.2500353 1.3808414 1.3805333 5.422494 ]

📊 VALIDATION:
Spend allocation difference: [0 0 0 0] (max: 0)
ROI difference: [7.12108612e-05 2.63620377e-03 4.44849014e-03 2.25170135e-03] (max: 0.004)
Execution time: 1.83 seconds
✅ Channel0: All metrics identical (point estimates)
✅ Channel1: All metrics identical (point estimates)
✅ Channel2: All metrics identical (point estimates)
✅ Channel3: All metrics identical (point estimates)
✅ Test 1.1 completed successfully


## Test 1.2: Posterior vs Prior Usage Validation

In [7]:
print("\nRunning Test 1.2: Posterior vs Prior Usage Validation")
print("Verifying that optimization uses posterior point estimates correctly.")

try:
    # Test with use_posterior=True (should use Excel estimates)
    dummy_model, budget_optimizer = setup_model_and_optimizer()
    opt_results_posterior = budget_optimizer.optimize(use_posterior=True)
    results_posterior = extract_optimization_results(opt_results_posterior, "Posterior")

    # Test with use_posterior=False (should use prior samples)
    dummy_model, budget_optimizer = setup_model_and_optimizer()
    opt_results_prior = budget_optimizer.optimize(use_posterior=False)
    results_prior = extract_optimization_results(opt_results_prior, "Prior")

    print_test_results(results_posterior, "Results with use_posterior=True")
    print_test_results(results_prior, "Results with use_posterior=False")

    # Compare results
    spend_diff = np.abs(results_posterior['spend'] - results_prior['spend'])
    roi_diff = np.abs(results_posterior['roi_mean'] - results_prior['roi_mean'])

    print(f"\n📊 COMPARISON:")
    print(f"Spend difference: {spend_diff} (max: {np.max(spend_diff):,.0f})")
    print(f"ROI difference: {roi_diff} (max: {np.max(roi_diff):.3f})")

    if np.max(spend_diff) > 1000000 or np.max(roi_diff) > 0.1:  # Significant difference
        print("✅ Posterior vs Prior produces different results (as expected)")
    else:
        print("⚠️ Posterior vs Prior produces similar results (unexpected)")

    test_results.extend([results_posterior, results_prior])
    print("✅ Test 1.2 completed successfully")

except Exception as e:
    print(f"❌ Test 1.2 failed: {str(e)}")
    test_results.append({'test_name': 'Test 1.2: Posterior vs Prior', 'status': 'FAILED', 'error': str(e)})


Running Test 1.2: Posterior vs Prior Usage Validation
Verifying that optimization uses posterior point estimates correctly.



Results with use_posterior=True
Total Budget: $110,900,000
Total Incremental Outcome: 345,741,440
Overall ROI: 3.118

Spend Allocation: [28300000 27100000 30000000 25500000]
Spend Percentages: [25.51848512 24.43642922 27.05139766 22.99368801]
ROI by Channel: [1.8300712 2.5073638 2.9244485 5.4222517]
mROI by Channel: [1.2500353 1.3808414 1.3805333 5.422494 ]

Results with use_posterior=False
Total Budget: $110,900,000
Total Incremental Outcome: 229,963,200
Overall ROI: 2.074

Spend Allocation: [39400000 25000000 21000000 25500000]
Spend Percentages: [35.52750225 22.54283138 18.93597836 22.99368801]
ROI by Channel: [2.0426452 1.988598  2.1256335 2.1619494]
mROI by Channel: [1.0022843 1.0014747 0.9997364 2.1619496]

📊 COMPARISON:
Spend difference: [11100000  2100000  9000000        0] (max: 11,100,000)
ROI difference: [0.212574  0.5187658 0.798815  3.2603023] (max: 3.260)
✅ Posterior vs Prior produces different results (as expected)
✅ Test 1.2 completed successfully


## Test 2.1: Fixed Budget Scenarios

In [10]:
print("\nRunning Test 2.1: Fixed Budget Scenarios")
print("Testing fixed budget optimization with different budget amounts.")

budget_scenarios = [
    (None, "Historical Budget (Default)"),
    (55_000_000, "50% of Historical Budget"),
    (165_000_000, "150% of Historical Budget"),
    (220_000_000, "200% of Historical Budget")
]

budget_results = []

for budget, scenario_name in budget_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")
        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(budget=budget, fixed_budget=True)
        results = extract_optimization_results(opt_results, scenario_name)

        budget_results.append(results)

        print(f"Target Budget: {budget if budget else 'Historical'}")
        print(f"Actual Total Spend: ${results['total_budget']:,.0f}")
        print(f"Overall ROI: {results['total_incremental_outcome']/results['total_budget']:.3f}")
        print(f"Spend Allocation: {results['pct_of_spend']*100}")

        if budget:
            budget_diff = abs(results['total_budget'] - budget)
            if budget_diff < 1000:  # Within $1K tolerance
                print(f"✅ Budget allocation accurate (diff: ${budget_diff:,.0f})")
            else:
                print(f"⚠️ Budget allocation off by ${budget_diff:,.0f}")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        budget_results.append({'test_name': scenario_name, 'status': 'FAILED', 'error': str(e)})

# Analysis across budget levels
if len(budget_results) > 1:
    print(f"\n📊 BUDGET SCALING ANALYSIS:")
    successful_results = [r for r in budget_results if 'status' not in r]

    for i, result in enumerate(successful_results):
        roi = result['total_incremental_outcome'] / result['total_budget']
        print(f"{result['test_name']}: Budget ${result['total_budget']:,.0f}, Outcome {result['total_incremental_outcome']:,.0f}, ROI {roi:.3f}")


test_results.extend(budget_results)
print("✅ Test 2.1 completed")


Running Test 2.1: Fixed Budget Scenarios
Testing fixed budget optimization with different budget amounts.

--- Historical Budget (Default) ---
Target Budget: Historical
Actual Total Spend: $110,900,000
Overall ROI: 3.118
Spend Allocation: [25.51848512 24.43642922 27.05139766 22.99368801]

--- 50% of Historical Budget ---
Target Budget: 55000000
Actual Total Spend: $55,000,000
Overall ROI: 3.678
Spend Allocation: [25.50909091 24.21818182 27.27272727 23.        ]
✅ Budget allocation accurate (diff: $0)

--- 150% of Historical Budget ---
Target Budget: 165000000
Actual Total Spend: $165,000,000
Overall ROI: 2.788
Spend Allocation: [25.87878788 24.78787879 26.3030303  23.03030303]
✅ Budget allocation accurate (diff: $0)

--- 200% of Historical Budget ---
Target Budget: 220000000
Actual Total Spend: $219,900,000
Overall ROI: 2.557
Spend Allocation: [27.23965439 24.37471578 25.37517053 23.0104593 ]
⚠️ Budget allocation off by $100,000

📊 BUDGET SCALING ANALYSIS:
Historical Budget (Default):

## Test 2.2: Flexible Budget with ROI Targets

In [18]:
print("\nRunning Test 2.2: Flexible Budget with ROI Targets")
print("Testing flexible budget optimization with different target ROI values.")

roi_scenarios = [
    (2.0, "Conservative Target"),
    (3.0, "Moderate Target"),
    (4.0, "Aggressive Target"),
    (6.0, "Potentially Unrealistic Target")
]
historical_budget = 110900000
roi_results = []

for target_roi, scenario_name in roi_scenarios:
    try:
        print(f"\n--- {scenario_name} (ROI: {target_roi}) ---")
        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            fixed_budget=False,
            target_roi=target_roi
        )
        results = extract_optimization_results(opt_results, f"ROI Target {target_roi}")

        actual_roi = results['total_incremental_outcome'] / results['total_budget']
        roi_diff = abs(actual_roi - target_roi)

        roi_results.append(results)

        print(f"Target ROI: {target_roi:.1f}")
        print(f"Achieved ROI: {actual_roi:.3f}")
        print(f"Required Budget: ${results['total_budget']:,.0f}")
        print(f"Required Budget change: {((results['total_budget'] - historical_budget)/historical_budget)* 100:.4f}%")
        print(f"ROI Difference: {roi_diff:.3f}")

        if roi_diff < 0.05:  # Within 5% tolerance
            print(f"✅ ROI target achieved accurately")
        elif roi_diff > 0:
            print(f"✅✅ Exceeded Target ROI goal!")
        else:
            print(f"⚠️ ROI target missed by {roi_diff:.3f}")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        roi_results.append({'test_name': f"ROI Target {target_roi}", 'status': 'FAILED', 'error': str(e)})

# Analysis of ROI vs Budget relationship
if len(roi_results) > 1:
    print(f"\n📊 ROI-BUDGET RELATIONSHIP:")
    successful_results = [r for r in roi_results if 'status' not in r]

    for result in successful_results:
        roi = result['total_incremental_outcome'] / result['total_budget']
        print(f"ROI {roi:.2f}: Budget ${result['total_budget']:,.0f}")

test_results.extend(roi_results)
print("✅ Test 2.2 completed")


Running Test 2.2: Flexible Budget with ROI Targets
Testing flexible budget optimization with different target ROI values.

--- Conservative Target (ROI: 2.0) ---
Target ROI: 2.0
Achieved ROI: 2.285
Required Budget: $221,800,000
Required Budget change: 100.0000%
ROI Difference: 0.285
✅✅ Exceeded Target ROI goal!

--- Moderate Target (ROI: 3.0) ---
Target ROI: 3.0
Achieved ROI: 3.000
Required Budget: $150,200,000
Required Budget change: 35.4373%
ROI Difference: 0.000
✅ ROI target achieved accurately

--- Aggressive Target (ROI: 4.0) ---
Target ROI: 4.0
Achieved ROI: 4.001
Required Budget: $92,400,000
Required Budget change: -16.6817%
ROI Difference: 0.001
✅ ROI target achieved accurately

--- Potentially Unrealistic Target (ROI: 6.0) ---
Target ROI: 6.0
Achieved ROI: 6.005
Required Budget: $1,800,000
Required Budget change: -98.3769%
ROI Difference: 0.005
✅ ROI target achieved accurately

📊 ROI-BUDGET RELATIONSHIP:
ROI 2.29: Budget $221,800,000
ROI 3.00: Budget $150,200,000
ROI 4.00: Bu

## Test 2.3: Flexible Budget with mROI Targets

In [19]:
print("\nRunning Test 2.3: Flexible Budget with mROI Targets")
print("Testing flexible budget optimization with different target marginal ROI values.")

mroi_scenarios = [
    (1.5, "High Efficiency Target"),
    (2.0, "Moderate Efficiency Target"),
    (3.0, "Conservative Efficiency Target")
]

mroi_results = []

for target_mroi, scenario_name in mroi_scenarios:
    try:
        print(f"\n--- {scenario_name} (mROI: {target_mroi}) ---")
        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            fixed_budget=False,
            target_roi=None,
            target_mroi=target_mroi
        )
        results = extract_optimization_results(opt_results, f"mROI Target {target_mroi}")

        # Check achieved mROI values
        achieved_mrois = results['mroi_mean']
        avg_mroi = np.mean(achieved_mrois)
        roi = results['total_incremental_outcome'] / results['total_budget']

        mroi_results.append(results)

        print(f"Target mROI: {target_mroi:.1f}")
        print(f"Achieved mROIs: {achieved_mrois}")
        print(f"Average mROI: {avg_mroi:.3f}")
        print(f"Overall ROI: {roi:.3f}")
        print(f"Required Budget: ${results['total_budget']:,.0f}")

        # Check if any channel achieved target mROI
        channels_at_target = np.sum(np.abs(achieved_mrois - target_mroi) < 0.1)
        print(f"Channels near target mROI: {channels_at_target}/4")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        mroi_results.append({'test_name': f"mROI Target {target_mroi}", 'status': 'FAILED', 'error': str(e)})

test_results.extend(mroi_results)
print("✅ Test 2.3 completed")


Running Test 2.3: Flexible Budget with mROI Targets
Testing flexible budget optimization with different target marginal ROI values.

--- High Efficiency Target (mROI: 1.5) ---
Target mROI: 1.5
Achieved mROIs: [1.494932  1.4942672 1.4931985 5.422408 ]
Average mROI: 2.476
Overall ROI: 3.589
Required Budget: $112,200,000
Channels near target mROI: 3/4

--- Moderate Efficiency Target (mROI: 2.0) ---
Target mROI: 2.0
Achieved mROIs: [1.9955417 1.9924524 1.9941492 5.422408 ]
Average mROI: 2.851
Overall ROI: 4.164
Required Budget: $85,700,000
Channels near target mROI: 3/4

--- Conservative Efficiency Target (mROI: 3.0) ---
Target mROI: 3.0
Achieved mROIs: [0.       2.998763 2.998386 5.422408]
Average mROI: 2.855
Overall ROI: 4.996
Required Budget: $58,200,000
Channels near target mROI: 2/4
✅ Test 2.3 completed


## Test 2.4: Custom Spend Allocation

In [20]:
print("\nRunning Test 2.4: Custom Spend Allocation")
print("Testing custom percentage spend allocation constraints.")

allocation_scenarios = [
    ([0.25, 0.25, 0.25, 0.25], "Equal Allocation"),
    ([0.50, 0.20, 0.20, 0.10], "Heavy Channel0"),
    ([0.15, 0.15, 0.20, 0.50], "Heavy Channel3"),
    ([0.40, 0.35, 0.0, 0.25], "No Channel2")
]

allocation_results = []

for pct_allocation, scenario_name in allocation_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")
        print(f"Target allocation: {[p*100 for p in pct_allocation]}%")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            pct_of_spend=pct_allocation
        )
        results = extract_optimization_results(opt_results, scenario_name)

        actual_allocation = results['pct_of_spend']
        allocation_diff = np.abs(actual_allocation - np.array(pct_allocation))
        roi = results['total_incremental_outcome'] / results['total_budget']

        allocation_results.append(results)

        print(f"Achieved allocation: {actual_allocation*100}%")
        print(f"Allocation difference: {allocation_diff*100}%")
        print(f"Overall ROI: {roi:.3f}")
        print(f"ROI by Channel: {results['roi_mean']}")

        if np.max(allocation_diff) < 0.01:  # Within 1% tolerance
            print(f"✅ Allocation constraint respected accurately")
        else:
            print(f"⚠️ Allocation constraint violated by {np.max(allocation_diff)*100:.1f}%")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        allocation_results.append({'test_name': scenario_name, 'status': 'FAILED', 'error': str(e)})

# Compare performance across allocations
if len(allocation_results) > 1:
    print(f"\n📊 ALLOCATION STRATEGY PERFORMANCE:")
    successful_results = [r for r in allocation_results if 'status' not in r]

    for result in successful_results:
        roi = result['total_incremental_outcome'] / result['total_budget']
        print(f"{result['test_name']:20}: ROI {roi:.3f}, Budget ${result['total_budget']:,.0f}")

test_results.extend(allocation_results)
print("✅ Test 2.4 completed")


Running Test 2.4: Custom Spend Allocation
Testing custom percentage spend allocation constraints.

--- Equal Allocation ---
Target allocation: [25.0, 25.0, 25.0, 25.0]%
Achieved allocation: [19.22382671 22.83393502 25.45126354 32.49097473]%
Allocation difference: [5.77617329 2.16606498 0.45126354 7.49097473]%
Overall ROI: 3.502
ROI by Channel: [1.9848065 2.5841417 3.0198185 5.422255 ]
⚠️ Allocation constraint violated by 7.5%

--- Heavy Channel0 ---
Target allocation: [50.0, 20.0, 20.0, 10.0]%
Achieved allocation: [34.95495495 26.03603604 26.03603604 12.97297297]%
Allocation difference: [15.04504505  6.03603604  6.03603604  2.97297297]%
Overall ROI: 2.687
ROI by Channel: [1.6389209 2.4351559 2.9819665 5.422254 ]
⚠️ Allocation constraint violated by 15.0%

--- Heavy Channel3 ---
Target allocation: [15.0, 15.0, 20.0, 50.0]%
Achieved allocation: [10.45987376 10.45987376 13.97655546 65.10369702]%
Allocation difference: [ 4.54012624  4.54012624  6.02344454 15.10369702]%
Overall ROI: 4.670


## Test 3.1: Partial Time Period Optimization

In [ ]:
print("\nRunning Test 3.1: Partial Time Period Optimization")
print("Testing optimization across different time periods.")

# Using actual dataset dates (2021-01-25 to 2024-01-15)
time_scenarios = [
    ('2021-01-25', '2021-10-18', 'First Quarter (39 weeks)'),
    ('2021-10-25', '2023-04-17', 'Middle Year (78 weeks)'),
    ('2023-07-24', '2024-01-15', 'Last 6 Months (26 weeks)'),
    ('2022-07-25', '2022-08-15', 'Single Month (4 weeks)')
]

time_results = []

def calculate_period_length_days(start_date, end_date):
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    return (end - start).days + 1

for start_date, end_date, scenario_name in time_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")
        print(f"Period: {start_date} to {end_date}")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            start_date=start_date,
            end_date=end_date,
            fixed_budget=True
        )
        results = extract_optimization_results(opt_results, scenario_name)

        period_days = calculate_period_length_days(start_date, end_date)
        budget_per_day = results['total_budget'] / period_days
        outcome_per_day = results['total_incremental_outcome'] / period_days
        roi = results['total_incremental_outcome'] / results['total_budget']

        results['period_days'] = period_days
        results['budget_per_day'] = budget_per_day
        results['outcome_per_day'] = outcome_per_day
        results['period_roi'] = roi

        time_results.append(results)

        print(f"Period Length: {period_days} days")
        print(f"Total Budget: ${results['total_budget']:,.0f}")
        print(f"Budget per Day: ${budget_per_day:,.0f}")
        print(f"ROI: {roi:.3f}")
        print(f"Spend Allocation: {results['pct_of_spend']*100}")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        time_results.append({'test_name': scenario_name, 'status': 'FAILED', 'error': str(e)})

# Seasonal analysis
if len(time_results) > 1:
    print(f"\n📊 SEASONAL EFFECTS ANALYSIS:")
    successful_results = [r for r in time_results if 'status' not in r]

    rois = [r['period_roi'] for r in successful_results]
    roi_range = max(rois) - min(rois)
    roi_mean = np.mean(rois)

    print(f"ROI Range: {roi_range:.3f} ({(roi_range/roi_mean*100):.1f}% of mean)")

    best_period = max(successful_results, key=lambda x: x['period_roi'])
    worst_period = min(successful_results, key=lambda x: x['period_roi'])

    print(f"Best ROI Period: {best_period['test_name']} (ROI: {best_period['period_roi']:.3f})")
    print(f"Worst ROI Period: {worst_period['test_name']} (ROI: {worst_period['period_roi']:.3f})")

    if roi_range > 0.1:
        print("⚠️ Significant seasonal variation detected!")
    else:
        print("✅ ROI relatively consistent across periods")

test_results.extend(time_results)
print("✅ Test 3.1 completed")

## Test 4.1: Lower Spend Constraints

In [ ]:
print("\nRunning Test 4.1: Lower Spend Constraints")
print("Testing optimization with minimum spend constraints.")

# Get baseline spending for comparison
dummy_model, budget_optimizer = setup_model_and_optimizer()
baseline_results = budget_optimizer.optimize()
baseline_spend = extract_optimization_results(baseline_results, "Baseline")['spend']
print(f"Baseline spend: {baseline_spend}")

constraint_scenarios = [
    (0.2, "Conservative (80% minimum)"),
    (0.5, "Moderate (50% minimum)"),
    (0.8, "Aggressive (20% minimum)"),
    ([0.2, 0.3, 0.5, 0.1], "Channel-specific")
]

constraint_results = []

for constraint, scenario_name in constraint_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")
        print(f"Constraint: {constraint}")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            spend_constraint_lower=constraint
        )
        results = extract_optimization_results(opt_results, scenario_name)

        # Calculate minimum allowed spend
        if isinstance(constraint, list):
            min_allowed = baseline_spend * (1 - np.array(constraint))
        else:
            min_allowed = baseline_spend * (1 - constraint)

        # Check constraint compliance
        constraint_violations = results['spend'] < min_allowed

        constraint_results.append(results)

        print(f"Minimum allowed spend: {min_allowed}")
        print(f"Actual spend: {results['spend']}")
        print(f"Constraint violations: {constraint_violations}")
        print(f"Overall ROI: {results['total_incremental_outcome']/results['total_budget']:.3f}")

        if not np.any(constraint_violations):
            print(f"✅ All lower constraints respected")
        else:
            print(f"❌ Constraint violations detected")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        constraint_results.append({'test_name': scenario_name, 'status': 'FAILED', 'error': str(e)})

test_results.extend(constraint_results)
print("✅ Test 4.1 completed")

## Test 4.2: Upper Spend Constraints

In [ ]:
print("\nRunning Test 4.2: Upper Spend Constraints")
print("Testing optimization with maximum spend constraints.")

upper_constraint_scenarios = [
    (0.2, "Conservative (120% maximum)"),
    (0.5, "Moderate (150% maximum)"),
    (1.0, "Aggressive (200% maximum)"),
    ([0.3, 0.2, 0.8, 0.5], "Channel-specific")
]

upper_constraint_results = []

for constraint, scenario_name in upper_constraint_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")
        print(f"Constraint: {constraint}")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            spend_constraint_upper=constraint
        )
        results = extract_optimization_results(opt_results, scenario_name)

        # Calculate maximum allowed spend
        if isinstance(constraint, list):
            max_allowed = baseline_spend * (1 + np.array(constraint))
        else:
            max_allowed = baseline_spend * (1 + constraint)

        # Check constraint compliance
        constraint_violations = results['spend'] > max_allowed

        upper_constraint_results.append(results)

        print(f"Maximum allowed spend: {max_allowed}")
        print(f"Actual spend: {results['spend']}")
        print(f"Constraint violations: {constraint_violations}")
        print(f"Overall ROI: {results['total_incremental_outcome']/results['total_budget']:.3f}")

        if not np.any(constraint_violations):
            print(f"✅ All upper constraints respected")
        else:
            print(f"❌ Constraint violations detected")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        upper_constraint_results.append({'test_name': scenario_name, 'status': 'FAILED', 'error': str(e)})

test_results.extend(upper_constraint_results)
print("✅ Test 4.2 completed")

---
# 🟢 P2: Advanced Configuration Tests

## Test 5.1: KPI vs Revenue Optimization

In [ ]:
print("\nRunning Test 5.1: KPI vs Revenue Optimization")
print("Comparing optimization for KPI vs Revenue focus.")

kpi_scenarios = [
    (False, "Revenue Optimization (default)"),
    (True, "KPI Optimization")
]

kpi_results = []

for use_kpi, scenario_name in kpi_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            use_kpi=use_kpi
        )
        results = extract_optimization_results(opt_results, scenario_name)

        kpi_results.append(results)

        roi = results['total_incremental_outcome'] / results['total_budget']
        print(f"Overall ROI: {roi:.3f}")
        print(f"Spend Allocation: {results['pct_of_spend']*100}")
        print(f"ROI by Channel: {results['roi_mean']}")
        print(f"mROI by Channel: {results['mroi_mean']}")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        kpi_results.append({'test_name': scenario_name, 'status': 'FAILED', 'error': str(e)})

# Compare KPI vs Revenue results
if len(kpi_results) == 2 and all('status' not in r for r in kpi_results):
    revenue_results, kpi_results_data = kpi_results

    print(f"\n📊 KPI vs REVENUE COMPARISON:")

    spend_diff = np.abs(revenue_results['spend'] - kpi_results_data['spend'])
    roi_diff = np.abs(revenue_results['roi_mean'] - kpi_results_data['roi_mean'])

    print(f"Spend allocation differences: {spend_diff}")
    print(f"ROI differences: {roi_diff}")

    revenue_roi = revenue_results['total_incremental_outcome'] / revenue_results['total_budget']
    kpi_roi = kpi_results_data['total_incremental_outcome'] / kpi_results_data['total_budget']

    print(f"Revenue focus ROI: {revenue_roi:.3f}")
    print(f"KPI focus ROI: {kpi_roi:.3f}")
    print(f"ROI difference: {abs(revenue_roi - kpi_roi):.3f}")

test_results.extend(kpi_results)
print("✅ Test 5.1 completed")

## Test 5.2: Optimal vs Historical Frequency

In [ ]:
print("\nRunning Test 5.2: Optimal vs Historical Frequency")
print("Testing impact of frequency optimization on reach channels (Channel3).")

frequency_scenarios = [
    (True, "Optimal Frequency (default)"),
    (False, "Historical Frequency")
]

frequency_results = []

for use_optimal_freq, scenario_name in frequency_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            use_optimal_frequency=use_optimal_freq
        )
        results = extract_optimization_results(opt_results, scenario_name)

        frequency_results.append(results)

        # Focus on Channel3 (R&F channel)
        channel3_roi = results['roi_mean'][3]
        channel3_spend_pct = results['pct_of_spend'][3] * 100

        print(f"Channel3 ROI: {channel3_roi:.3f}")
        print(f"Channel3 Spend %: {channel3_spend_pct:.1f}%")
        print(f"Overall ROI: {results['total_incremental_outcome']/results['total_budget']:.3f}")

    except Exception as e:
        print(f"❌ {scenario_name} failed: {str(e)}")
        frequency_results.append({'test_name': scenario_name, 'status': 'FAILED', 'error': str(e)})

# Compare frequency optimization impact
if len(frequency_results) == 2 and all('status' not in r for r in frequency_results):
    optimal_results, historical_results = frequency_results

    print(f"\n📊 FREQUENCY OPTIMIZATION IMPACT:")

    optimal_ch3_roi = optimal_results['roi_mean'][3]
    historical_ch3_roi = historical_results['roi_mean'][3]
    roi_lift = ((optimal_ch3_roi - historical_ch3_roi) / historical_ch3_roi) * 100

    print(f"Channel3 ROI - Optimal: {optimal_ch3_roi:.3f}")
    print(f"Channel3 ROI - Historical: {historical_ch3_roi:.3f}")
    print(f"Performance lift: {roi_lift:.1f}%")

    # Check if only reach channels are affected
    impression_channel_diffs = np.abs(optimal_results['roi_mean'][:3] - historical_results['roi_mean'][:3])
    max_impression_diff = np.max(impression_channel_diffs)

    if max_impression_diff < 0.01:
        print(f"✅ Impression channels unaffected (max diff: {max_impression_diff:.4f})")
    else:
        print(f"⚠️ Impression channels affected (max diff: {max_impression_diff:.4f})")

test_results.extend(frequency_results)
print("✅ Test 5.2 completed")

---
# 🟢 P3: Edge Case Tests

## Test 6.1: Extreme Budget Values

In [ ]:
print("\nRunning Test 6.1: Extreme Budget Values")
print("Testing optimization with extreme budget values.")

extreme_scenarios = [
    (1000, "Very Low Budget"),
    (1_000_000_000, "Very High Budget"),
    (0, "Zero Budget"),
    (-100000, "Negative Budget")
]

extreme_results = []

for budget, scenario_name in extreme_scenarios:
    try:
        print(f"\n--- {scenario_name} (Budget: {budget:,}) ---")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(
            budget=budget,
            fixed_budget=True
        )
        results = extract_optimization_results(opt_results, scenario_name)

        extreme_results.append(results)

        roi = results['total_incremental_outcome'] / results['total_budget'] if results['total_budget'] > 0 else 0
        print(f"✅ Optimization completed")
        print(f"Total Budget: ${results['total_budget']:,.0f}")
        print(f"Overall ROI: {roi:.3f}")
        print(f"Spend Allocation: {results['pct_of_spend']*100}")

        # For very low budgets, check if highest ROI channels get priority
        if budget <= 10000:
            max_roi_channel = np.argmax(results['roi_mean'])
            max_spend_channel = np.argmax(results['spend'])
            if max_roi_channel == max_spend_channel:
                print(f"✅ Highest ROI channel prioritized (Channel{max_roi_channel})")
            else:
                print(f"⚠️ Highest ROI channel not prioritized")

    except Exception as e:
        print(f"⚠️ {scenario_name} failed (expected for invalid budgets): {str(e)}")
        extreme_results.append({'test_name': scenario_name, 'status': 'FAILED_EXPECTED', 'error': str(e)})

# Check error handling for invalid budgets
failed_count = sum(1 for r in extreme_results if 'status' in r and 'FAILED' in r['status'])
print(f"\n📊 EDGE CASE HANDLING: {failed_count}/4 extreme cases failed (expected for zero/negative budgets)")

test_results.extend(extreme_results)
print("✅ Test 6.1 completed")

## Test 6.2: Invalid Parameter Combinations

In [ ]:
print("\nRunning Test 6.2: Invalid Parameter Combinations")
print("Testing error handling for invalid parameter combinations.")

invalid_scenarios = [
    ({"fixed_budget": False, "target_roi": None, "target_mroi": None},
     "Flexible budget with no target"),
    ({"target_roi": 2.0, "target_mroi": 1.5},
     "Both ROI targets specified"),
    ({"start_date": '2025-01-01'},
     "Future date"),
    ({"start_date": '2022-01-01', "end_date": '2021-01-01'},
     "Backwards dates"),
    ({"pct_of_spend": [0.3, 0.3, 0.3]},
     "Invalid allocation (doesn't sum to 1.0)")
]

invalid_results = []

for params, scenario_name in invalid_scenarios:
    try:
        print(f"\n--- {scenario_name} ---")
        print(f"Parameters: {params}")

        dummy_model, budget_optimizer = setup_model_and_optimizer()

        opt_results = budget_optimizer.optimize(**params)
        results = extract_optimization_results(opt_results, scenario_name)

        print(f"⚠️ Unexpected success - should have failed")
        invalid_results.append(results)

    except Exception as e:
        print(f"✅ Properly failed with error: {type(e).__name__}: {str(e)[:100]}...")
        invalid_results.append({'test_name': scenario_name, 'status': 'FAILED_EXPECTED', 'error': str(e)})

# Count expected failures
expected_failures = sum(1 for r in invalid_results if 'status' in r and 'FAILED_EXPECTED' in r['status'])
print(f"\n📊 ERROR HANDLING: {expected_failures}/{len(invalid_scenarios)} invalid scenarios properly rejected")

test_results.extend(invalid_results)
print("✅ Test 6.2 completed")

---
# 📊 Test Results Summary

In [ ]:
# Update todo status
print("\n" + "="*80)
print("FAST OPTIMIZATION TEST SUITE SUMMARY")
print("="*80)

# Count test outcomes
total_tests = len(test_results)
successful_tests = sum(1 for r in test_results if 'status' not in r)
failed_tests = sum(1 for r in test_results if 'status' in r and 'FAILED' in r['status'] and 'EXPECTED' not in r['status'])
expected_failures = sum(1 for r in test_results if 'status' in r and 'EXPECTED' in r['status'])

print(f"\n📈 TEST EXECUTION SUMMARY:")
print(f"Total Tests Run: {total_tests}")
print(f"Successful: {successful_tests}")
print(f"Failed: {failed_tests}")
print(f"Expected Failures (Edge Cases): {expected_failures}")
print(f"Success Rate: {(successful_tests/(total_tests-expected_failures)*100):.1f}%")

# Performance analysis
if any('execution_time' in r for r in test_results if 'status' not in r):
    execution_times = [r['execution_time'] for r in test_results if 'execution_time' in r]
    avg_time = np.mean(execution_times)
    max_time = np.max(execution_times)

    print(f"\n⏱️ PERFORMANCE METRICS:")
    print(f"Average execution time: {avg_time:.2f} seconds")
    print(f"Maximum execution time: {max_time:.2f} seconds")
    print(f"Total test suite runtime: {sum(execution_times):.2f} seconds")

# ROI consistency analysis
successful_roi_tests = [r for r in test_results if 'status' not in r and 'total_budget' in r and r['total_budget'] > 0]
if len(successful_roi_tests) > 5:
    rois = [r['total_incremental_outcome']/r['total_budget'] for r in successful_roi_tests]
    roi_std = np.std(rois)
    roi_mean = np.mean(rois)
    roi_range = max(rois) - min(rois)

    print(f"\n🎯 ROI CONSISTENCY ANALYSIS:")
    print(f"ROI Mean: {roi_mean:.3f}")
    print(f"ROI Std Dev: {roi_std:.3f}")
    print(f"ROI Range: {roi_range:.3f}")
    print(f"Coefficient of Variation: {(roi_std/roi_mean)*100:.1f}%")

# Failed tests details
if failed_tests > 0:
    print(f"\n❌ FAILED TESTS:")
    for r in test_results:
        if 'status' in r and 'FAILED' in r['status'] and 'EXPECTED' not in r['status']:
            print(f"  - {r['test_name']}: {r.get('error', 'Unknown error')}")

# Test coverage summary
print(f"\n✅ TEST COVERAGE ACHIEVED:")
print(f"  🔴 P0 Basic Functionality: Default parameters, posterior vs prior")
print(f"  🟡 P1 Budget Configurations: Fixed budgets, ROI/mROI targets, custom allocation")
print(f"  🟡 P1 Time Period Tests: Partial time periods, seasonal analysis")
print(f"  🟢 P2 Constraint Tests: Lower/upper spend constraints")
print(f"  🟢 P2 Advanced Configurations: KPI vs revenue, frequency optimization")
print(f"  🟢 P3 Edge Cases: Extreme budgets, invalid parameters")

print(f"\n🏆 FRAMEWORK VALIDATION COMPLETE!")
if failed_tests == 0:
    print(f"   ✅ All critical tests passed - Framework is ready for production use")
else:
    print(f"   ⚠️ {failed_tests} tests failed - Review and fix issues before production use")

print(f"\n📋 Next Steps:")
print(f"   1. Review any failed tests and fix underlying issues")
print(f"   2. Execute P4 performance benchmarking tests if needed")
print(f"   3. Deploy framework with confidence based on comprehensive validation")

---
# 🔍 Detailed Results DataFrame

In [ ]:
# Create detailed results DataFrame for analysis
detailed_results = []

for result in test_results:
    if 'status' not in result:  # Successful tests only
        row = {
            'Test_Name': result['test_name'],
            'Total_Budget': result['total_budget'],
            'Total_Outcome': result['total_incremental_outcome'],
            'Overall_ROI': result['total_incremental_outcome'] / result['total_budget'] if result['total_budget'] > 0 else 0,
            'Channel0_Spend': result['spend'][0],
            'Channel1_Spend': result['spend'][1],
            'Channel2_Spend': result['spend'][2],
            'Channel3_Spend': result['spend'][3],
            'Channel0_ROI': result['roi_mean'][0],
            'Channel1_ROI': result['roi_mean'][1],
            'Channel2_ROI': result['roi_mean'][2],
            'Channel3_ROI': result['roi_mean'][3],
            'Channel0_Pct': result['pct_of_spend'][0] * 100,
            'Channel1_Pct': result['pct_of_spend'][1] * 100,
            'Channel2_Pct': result['pct_of_spend'][2] * 100,
            'Channel3_Pct': result['pct_of_spend'][3] * 100
        }
        detailed_results.append(row)

if detailed_results:
    results_df = pd.DataFrame(detailed_results)

    # Display summary statistics
    print("\n📊 DETAILED RESULTS SUMMARY:")
    print("\nBudget and ROI Statistics:")
    summary_cols = ['Total_Budget', 'Total_Outcome', 'Overall_ROI']
    print(results_df[summary_cols].describe().round(2))

    print("\nChannel ROI Statistics:")
    roi_cols = ['Channel0_ROI', 'Channel1_ROI', 'Channel2_ROI', 'Channel3_ROI']
    print(results_df[roi_cols].describe().round(3))

    print("\nSpend Allocation Consistency:")
    pct_cols = ['Channel0_Pct', 'Channel1_Pct', 'Channel2_Pct', 'Channel3_Pct']
    print("Standard Deviations (lower = more consistent):")
    for col in pct_cols:
        std_dev = results_df[col].std()
        print(f"  {col}: {std_dev:.2f}%")

    # Save results for further analysis
    results_df.to_csv('fast_optimization_test_results.csv', index=False)
    print(f"\n💾 Detailed results saved to: fast_optimization_test_results.csv")

    # Display first few rows
    print(f"\n📋 Sample Results (first 5 tests):")
    display_cols = ['Test_Name', 'Total_Budget', 'Overall_ROI', 'Channel0_Pct', 'Channel1_Pct', 'Channel2_Pct', 'Channel3_Pct']
    print(results_df[display_cols].head().to_string(index=False))

else:
    print("\n⚠️ No successful test results to display")

print(f"\n\n🎉 Test suite execution completed successfully!")
print(f"📝 Review the results above and the saved CSV file for detailed analysis.")